# PKU YOLOv8 Detection Baseline in Google Colab

This notebook is for the **PKU COCO detection baseline only**.

- Model: `yolov8n.pt`
- Task: object detection
- Training style: fine-tuning / transfer learning from pretrained weights
- Dataset: `configs/pku_coco_baseline.yaml`
- Scope: PKU only, no DeepPCB, no tiling, no segmentation, no severity yet


## Colab Setup Notes

Use one of these project access options:

1. Put the whole repo in Google Drive and mount Drive in Colab.
2. Clone your GitHub repo into `/content/`.
3. Upload the project folder manually if needed.

The notebook below assumes **Google Drive** by default because it is the safest way to keep datasets, runs, and weights between sessions.


In [1]:
!pip install -q ultralytics==8.4.14 opencv-python pyyaml matplotlib

import platform
import sys
import torch

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    !nvidia-smi
else:
    print('GPU not detected. In Colab, switch Runtime > Change runtime type > GPU before training.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.8 MB/s eta 0:00:0000:01
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Torch: 2.10.0+cu128
CUDA available: True
Thu Apr 23 17:04:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N

## Mount Google Drive and Open the Repo

Update `PROJECT_ROOT` if your repo folder has a different Drive location.

If you prefer to clone the repo instead of using Drive, skip the mount lines and set `PROJECT_ROOT` to the cloned folder path.


In [2]:
from pathlib import Path

USE_DRIVE = True
PROJECT_ROOT = Path('/content/drive/MyDrive/PCB-Defect-Detector')  # change if needed

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

assert PROJECT_ROOT.exists(), f'Update PROJECT_ROOT to your repo folder: {PROJECT_ROOT}'

%cd $PROJECT_ROOT
print('Working directory:', PROJECT_ROOT)


Mounted at /content/drive
/content/drive/MyDrive/PCB-Defect-Detector
Working directory: /content/drive/MyDrive/PCB-Defect-Detector


## Prepare the PKU YOLO Workspace

This runs the repo's existing PKU preparation logic using the validated COCO config.
It does **not** start training yet.


In [3]:
!python scripts/pku_yolov8_colab_train.py --prepare-only
!cat configs/pku_yolov8_baseline_data.yaml


CONFIG=/content/drive/MyDrive/PCB-Defect-Detector/configs/pku_coco_baseline.yaml
WORKSPACE=/content/drive/MyDrive/PCB-Defect-Detector/data/yolo_ready/pku_yolov8_baseline
DATA_YAML=/content/drive/MyDrive/PCB-Defect-Detector/configs/pku_yolov8_baseline_data.yaml
MODEL=yolov8n.pt
TRAIN_IMAGES=2910
TRAIN_ANNOTATIONS=12187
TRAIN_MISSING_IMAGE_REFS=0
TRAIN_MISSING_IMAGE_FILES=0
TRAIN_INVALID_CATEGORY_IDS=0
TRAIN_MALFORMED_BBOX=0
TRAIN_NONPOSITIVE_BBOX=0
TRAIN_OUT_OF_BOUNDS_BBOX=0
VAL_IMAGES=276
VAL_ANNOTATIONS=1171
VAL_MISSING_IMAGE_REFS=0
VAL_MISSING_IMAGE_FILES=0
VAL_INVALID_CATEGORY_IDS=0
VAL_MALFORMED_BBOX=0
VAL_NONPOSITIVE_BBOX=0
VAL_OUT_OF_BOUNDS_BBOX=0
TEST_IMAGES=140
TEST_ANNOTATIONS=590
TEST_MISSING_IMAGE_REFS=0
TEST_MISSING_IMAGE_FILES=0
TEST_INVALID_CATEGORY_IDS=0
TEST_MALFORMED_BBOX=0
TEST_NONPOSITIVE_BBOX=0
TEST_OUT_OF_BOUNDS_BBOX=0
STATUS=PKU YOLO workspace and training data YAML are ready for Colab.
path: data/yolo_ready/pku_yolov8_baseline
train: images/train
val: images/val


## Baseline Training Settings

This notebook is now set for a longer PKU-only baseline run while keeping the same dataset, config, and workspace logic.


In [4]:
MODEL = 'yolov8n.pt'
EPOCHS = 50
IMGSZ = 640
BATCH = 16
WORKERS = 2
DEVICE = '0'
FRACTION = 1.0
PATIENCE = 20
PROJECT_DIR = 'runs/pku_baseline'
RUN_NAME = 'yolov8n_pku_baseline_colab_long50'

print({
    'model': MODEL,
    'epochs': EPOCHS,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'workers': WORKERS,
    'device': DEVICE,
    'fraction': FRACTION,
    'patience': PATIENCE,
    'project_dir': PROJECT_DIR,
    'run_name': RUN_NAME,
})


{'model': 'yolov8n.pt', 'epochs': 50, 'imgsz': 640, 'batch': 16, 'workers': 2, 'device': '0', 'fraction': 1.0, 'patience': 20, 'project_dir': 'runs/pku_baseline', 'run_name': 'yolov8n_pku_baseline_colab_long50'}


In [5]:
import subprocess

train_cmd = [
    'python', 'scripts/pku_yolov8_colab_train.py',
    '--model', MODEL,
    '--epochs', str(EPOCHS),
    '--imgsz', str(IMGSZ),
    '--batch', str(BATCH),
    '--workers', str(WORKERS),
    '--device', DEVICE,
    '--fraction', str(FRACTION),
    '--patience', str(PATIENCE),
    '--project', PROJECT_DIR,
    '--name', RUN_NAME,
]

print('Running:', ' '.join(train_cmd))
subprocess.run(train_cmd, check=True)


Running: python scripts/pku_yolov8_colab_train.py --model yolov8n.pt --epochs 50 --imgsz 640 --batch 16 --workers 2 --device 0 --fraction 1.0 --patience 20 --project runs/pku_baseline --name yolov8n_pku_baseline_colab_long50


CompletedProcess(args=['python', 'scripts/pku_yolov8_colab_train.py', '--model', 'yolov8n.pt', '--epochs', '50', '--imgsz', '640', '--batch', '16', '--workers', '2', '--device', '0', '--fraction', '1.0', '--patience', '20', '--project', 'runs/pku_baseline', '--name', 'yolov8n_pku_baseline_colab_long50'], returncode=0)

## Check the Training Outputs

Ultralytics may place the run under `runs/detect/...`, so this cell checks both likely output locations.


In [6]:
from pathlib import Path

candidate_run_dirs = [
    Path(PROJECT_DIR) / RUN_NAME,
    Path('runs/detect') / PROJECT_DIR / RUN_NAME,
]

RUN_DIR = None
for candidate in candidate_run_dirs:
    if candidate.exists():
        RUN_DIR = candidate
        break

assert RUN_DIR is not None, 'Training run folder not found yet.'

print('Run directory:', RUN_DIR)
print('Best weights:', RUN_DIR / 'weights' / 'best.pt')
print('Last weights:', RUN_DIR / 'weights' / 'last.pt')

results_csv = RUN_DIR / 'results.csv'
if results_csv.exists():
    import pandas as pd
    display(pd.read_csv(results_csv).tail())


Run directory: runs/detect/runs/pku_baseline/yolov8n_pku_baseline_colab_long50
Best weights: runs/detect/runs/pku_baseline/yolov8n_pku_baseline_colab_long50/weights/best.pt
Last weights: runs/detect/runs/pku_baseline/yolov8n_pku_baseline_colab_long50/weights/last.pt


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
45,46,3537.15,2.69393,3.00070,1.23743,0.12332,0.06035,0.04516,0.01633,2.85821,2.99327,1.20309,0.000109,0.000109,0.000109
46,47,3593.55,2.66827,2.96266,1.22802,0.11688,0.05772,0.04318,0.01598,2.84468,2.96550,1.20180,0.000089,0.000089,0.000089
47,48,3648.49,2.69041,2.95791,1.23128,0.14740,0.06509,0.04352,0.01593,2.82721,2.96122,1.19761,0.000069,0.000069,0.000069
48,49,3704.19,2.67135,2.93938,1.22151,0.13631,0.06014,0.04263,0.01624,2.81212,2.95921,1.19575,0.000050,0.000050,0.000050
49,50,3760.49,2.65213,2.93342,1.22766,0.12634,0.06582,0.04391,0.01627,2.81502,2.96187,1.19432,0.000030,0.000030,0.000030


## Quick Validation / Inference Check

This saves a few predicted validation images to a clean inspection folder and displays several inline.


In [7]:
from pathlib import Path
from IPython.display import Image, display
from ultralytics import YOLO

best_weights = RUN_DIR / 'weights' / 'best.pt'
assert best_weights.exists(), f'Missing best weights: {best_weights}'

val_dir = Path('data/resources/PCB Defects Detection.v1-pku-market-pcb-ver1.coco/valid')
sample_images = sorted(val_dir.glob('*.jpg'))[:8]
assert sample_images, f'No validation images found in {val_dir}'

pred_root = Path('data/inspection_outputs')
pred_name = f'{RUN_NAME}_predictions'

model = YOLO(str(best_weights))
results = model.predict(
    source=[str(path) for path in sample_images],
    imgsz=IMGSZ,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=str(pred_root),
    name=pred_name,
    exist_ok=True,
    verbose=False,
)

pred_dir = pred_root / pred_name
print('Prediction directory:', pred_dir)
print('Predictions generated for', len(results), 'images')

for image_path in sorted(pred_dir.glob('*.jpg'))[:4]:
    display(Image(filename=str(image_path)))


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Results saved to /content/drive/MyDrive/PCB-Defect-Detector/runs/detect/data/inspection_outputs/yolov8n_pku_baseline_colab_long50_predictions
Prediction directory: data/inspection_outputs/yolov8n_pku_baseline_colab_long50_predictions
Predictions generated for 8 images


## Notes

- If the first short Colab run gives weak predictions, that is normal.
- The point of this notebook is to give you a clean PKU-only baseline training workflow in Colab.
- DeepPCB training, dataset merging, tiling, segmentation, severity, and augmentation stay for later steps.
